# Regular Expressions (Regex)

Search and transform text with Python’s `re` module.

> 📘 **Learning path:** Python Foundations · Notebook 15 of 18

## Learning objectives

- Recognize common regex metacharacters.
- Choose between `search()`, `match()`, and `fullmatch()`.
- Extract named and numbered groups.
- Replace and split text with patterns.
- Understand the limitations of regex-based validation.

## How to use this notebook

1. Read each explanation before running its code cell.
2. Run the examples in order and change the values to observe what happens.
3. Complete the practice tasks before opening the suggested solutions at the end.

**Requirements:** Python 3.10+ and a Jupyter-compatible environment. The examples use the Python standard library unless stated otherwise.

## 1. What is a regular expression?

A regular expression (regex) describes a text pattern. Python's `re` module provides tools to search, validate, split, and replace text using patterns. Regex is powerful, but a complicated expression can be hard to maintain; use simple string methods when they are enough.

## 2. Common pattern symbols

| Pattern | Meaning |
|---|---|
| `.` | Any character except newline by default |
| `\d` | A digit |
| `\w` | A word character |
| `\s` | Whitespace |
| `+` | One or more repetitions |
| `*` | Zero or more repetitions |
| `?` | Zero or one repetition |
| `{m,n}` | Between `m` and `n` repetitions |
| `^` / `$` | Start / end of string (with some mode-specific details) |
| `[abc]` | One character from a set |
| `( ... )` | Capturing group |

Use raw strings such as `r"\d+"` to avoid Python string-escape confusion.

In [1]:
import re

text = "Order 104 contains 3 items; order 208 contains 12."
print(re.findall(r"\d+", text))
print(re.findall(r"[A-Za-z]+", text))

['104', '3', '208', '12']
['Order', 'contains', 'items', 'order', 'contains']


## 3. `search()`, `match()`, and `fullmatch()`

- `re.search()` finds the first match anywhere in the text.
- `re.match()` checks for a match at the start.
- `re.fullmatch()` requires the entire string to match the pattern.

These functions return match objects or `None`, so checking the result before using it is important.

In [2]:
code_pattern = r"[A-Z]{2}-\d{3}"
value = "AB-104"
print(bool(re.fullmatch(code_pattern, value)))
print(bool(re.match(r"\d+", "123abc")))
print(bool(re.fullmatch(r"\d+", "123abc")))

True
True
False


## 4. Capture groups

Parentheses define capture groups. Access groups through the match object's `.group()` method. Named groups such as `(?P<year>\d{4})` can make extracted fields easier to understand.

In [3]:
record = "Site: BAK-204 | Year: 2026"
match = re.search(r"Site: (?P<site>[A-Z]{3}-\d{3}) \| Year: (?P<year>\d{4})", record)
if match:
    print(match.group("site"))
    print(match.group("year"))

BAK-204
2026


## 5. Replace and split text

`re.sub()` replaces matching text, and `re.split()` divides a string around matches. When using substitutions, be clear about whether every occurrence should change or only a limited number.

In [4]:
sentence = "Room 402 is next to room 18."
print(re.sub(r"\d", "X", sentence))
print(re.split(r"[,;\s]+", "alpha, beta; gamma  delta"))

Room XXX is next to room XX.
['alpha', 'beta', 'gamma', 'delta']


## 6. Flags and case-insensitive patterns

Flags modify matching behavior. `re.IGNORECASE` makes matching case-insensitive. `re.MULTILINE` changes how `^` and `$` behave across multiple lines. `re.DOTALL` allows `.` to match newline characters. Use flags explicitly when they make the intention clearer.

In [5]:
text = "Python\nPYTHON\npython"
matches = re.findall(r"python", text, flags=re.IGNORECASE)
print(matches)

['Python', 'PYTHON', 'python']


## 7. Validation and limitations

Regex is useful for validating predictable formats, but it is not always the right tool for fully validating complex standards. For example, a simple email-like pattern can check a basic shape but cannot reliably determine whether an address exists or satisfies every rule. Prefer a well-maintained library or confirmation workflow for production validation of complex formats.

In [6]:
def is_simple_project_code(value):
    return re.fullmatch(r"[A-Z]{2}-\d{4}", value) is not None

for candidate in ("AB-1234", "ab-1234", "ABC-1234", "AB-12"):
    print(candidate, is_simple_project_code(candidate))

AB-1234 True
ab-1234 False
ABC-1234 False
AB-12 False


## Common mistakes

- Use raw strings for regex patterns containing backslashes.
- `search()` does not require the whole string to match; `fullmatch()` does.
- Greedy quantifiers such as `.*` can match more text than expected.
- Avoid building one enormous regex when several smaller steps are easier to test.

## 7. Choose the right matching function

- `re.search()` finds the first match anywhere in the string.
- `re.match()` checks only at the beginning.
- `re.fullmatch()` requires the entire string to match.

For validating an entire code or identifier, `fullmatch()` is often the clearest choice.

In [7]:
import re

pattern = r"[A-Z]{2}-\d{3}"
text = "Reference: AB-104"
print(re.search(pattern, text).group())
print(re.match(pattern, text))
print(bool(re.fullmatch(pattern, "AB-104")))

AB-104
None
True


## 8. Capture groups and match details

Parentheses create capture groups. `group(0)` is the complete match; `group(1)` is the first captured group. `finditer()` yields match objects that also provide start and end positions.

In [8]:
pattern = re.compile(r"(?P<code>[A-Z]{2})-(?P<number>\d{3})")
for match in pattern.finditer("AB-104 and XY-208"):
    print(match.group(0), match.group("code"), match.group("number"), match.span())

AB-104 AB 104 (0, 6)
XY-208 XY 208 (11, 17)


## 9. Escape literal text and understand limitations

Use `re.escape()` when a user-supplied string should be matched literally rather than treated as regex syntax. Regular expressions are useful for patterns, but a regex is not a complete substitute for domain-specific validation, such as checking whether a date exists or whether an email address can actually receive mail.

In [9]:
import re

literal = "price (USD)"
pattern = re.escape(literal)
print(bool(re.search(pattern, "Total price (USD): 25")))
print(re.sub(r"\s+", " ", "too   many\nspaces"))

True
too many spaces


### Greedy matching and flags

Quantifiers such as `.*` are greedy by default and match as much as possible. Add `?` to make a quantifier non-greedy when appropriate. Flags such as `re.IGNORECASE` can change matching behavior.

In [10]:
text = "<a>one</a><a>two</a>"
print(re.findall(r"<a>.*</a>", text))
print(re.findall(r"<a>.*?</a>", text))
print(bool(re.fullmatch(r"python", "PYTHON", flags=re.IGNORECASE)))

['<a>one</a><a>two</a>']
['<a>one</a>', '<a>two</a>']
True


## 🧩 Practice tasks

Try to solve these tasks without looking at the solutions. Write your attempts in new code cells below each task.

1. Extract all digit sequences from `"Block 12, parcel 104, floor 3"`.
2. Validate a code that consists of two uppercase letters, a hyphen, and four digits.
3. Extract a four-digit year from `"Report year: 2025"`.
4. Replace one or more whitespace characters in `"one   two\nthree"` with a single space.

## ✅ Suggested solutions

These are sample solutions, not the only correct answers. Try the exercises first, then compare your approach.

In [11]:
import re

text = "Case IDs: AB-104, XY-208, and CD-317."
pattern = r"[A-Z]{2}-\d{3}"
print(re.findall(pattern, text))
print(re.sub(r"\d", "X", "Room 402"))
print(bool(re.fullmatch(r"\d{4}", "2026")))

['AB-104', 'XY-208', 'CD-317']
Room XXX
True


## Key takeaways

- Focus on understanding what the code does, not memorizing every example.
- Make small changes and predict the result before running the cell.
- When an error appears, read its type and message; debugging is part of learning.

**Next notebook:** File Handling.